In [13]:
import os
import cv2
import xml.etree.ElementTree as ET
import numpy as np

In [20]:
# Input paths
video_path = r"E:\DATA\Videos\NO20251114-200552-143840F.MP4"
xml_path = r"E:\DATA\Annotations\NO20251114-200552-143840F.xml"

# Output path (saves in the same folder with "_annotated" suffix)
output_path = video_path.replace(".MP4", "_annotated.mp4")

print(f"Input Video: {video_path}")
print(f"Input XML: {xml_path}")
print(f"Output Video will be saved to: {output_path}")

Input Video: E:\DATA\Videos\NO20251114-200552-143840F.MP4
Input XML: E:\DATA\Annotations\NO20251114-200552-143840F.xml
Output Video will be saved to: E:\DATA\Videos\NO20251114-200552-143840F_annotated.mp4


In [21]:
def parse_cvat_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Dictionary to store annotations: { frame_number: [list_of_boxes] }
    frame_annotations = {}

    # Iterate through all 'track' elements (objects tracked across frames)
    for track in root.findall('track'):
        label = track.get('label')
        
        # Iterate through 'box' elements inside the track
        for box in track.findall('box'):
            frame_num = int(box.get('frame'))
            
            # Extract coordinates (CVAT stores them as floats, we need ints)
            xtl = int(float(box.get('xtl'))) # x top left
            ytl = int(float(box.get('ytl'))) # y top left
            xbr = int(float(box.get('xbr'))) # x bottom right
            ybr = int(float(box.get('ybr'))) # y bottom right
            
            # Check if the box is not marked as 'outside' (invisible)
            if box.get('outside') == '0':
                if frame_num not in frame_annotations:
                    frame_annotations[frame_num] = []
                
                frame_annotations[frame_num].append({
                    'label': label,
                    'coords': (xtl, ytl, xbr, ybr)
                })
                
    return frame_annotations

# Run the parser
annotations = parse_cvat_xml(xml_path)
print(f"Loaded annotations for {len(annotations)} frames.")

Loaded annotations for 4811 frames.


In [22]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: Could not open video.")
else:
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Initialize Video Writer
    # 'mp4v' is a standard codec for MP4 on Windows
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    current_frame = 0
    
    print("Processing video... (Press 'q' in the popup window to cancel)")

    while True:
        ret, frame = cap.read()
        
        if not ret:
            break # End of video
        
        # Check if there are annotations for this frame
        if current_frame in annotations:
            for obj in annotations[current_frame]:
                label = obj['label']
                x1, y1, x2, y2 = obj['coords']
                
                # Draw Rectangle (Green)
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # Draw Label background and text
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

        # Write the frame to the output video file
        out.write(frame)
        
        # --- PLAYBACK SECTION ---
        # Resize for display if video is 4K/too large (Optional, divides size by 2)
        display_frame = cv2.resize(frame, (width // 2, height // 2))
        cv2.imshow('Annotated Video Preview', display_frame)

        # Press 'q' to quit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Processing interrupted by user.")
            break
            
        current_frame += 1
        
        # Optional: Print progress every 100 frames
        if current_frame % 100 == 0:
            print(f"Processed {current_frame}/{total_frames} frames...")

    # Release resources
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    print("Done!")
    print(f"Video saved successfully at: {output_path}")

Processing video... (Press 'q' in the popup window to cancel)
Processed 100/5400 frames...
Processed 200/5400 frames...
Processed 300/5400 frames...
Processed 400/5400 frames...
Processed 500/5400 frames...
Processed 600/5400 frames...
Processed 700/5400 frames...
Processed 800/5400 frames...
Processed 900/5400 frames...
Processed 1000/5400 frames...
Processed 1100/5400 frames...
Processed 1200/5400 frames...
Processed 1300/5400 frames...
Processed 1400/5400 frames...
Processed 1500/5400 frames...
Processed 1600/5400 frames...
Processed 1700/5400 frames...
Processed 1800/5400 frames...
Processed 1900/5400 frames...
Processed 2000/5400 frames...
Processed 2100/5400 frames...
Processed 2200/5400 frames...
Processed 2300/5400 frames...
Processed 2400/5400 frames...
Processed 2500/5400 frames...
Processed 2600/5400 frames...
Processed 2700/5400 frames...
Processed 2800/5400 frames...
Processed 2900/5400 frames...
Processed 3000/5400 frames...
Processed 3100/5400 frames...
Processed 3200/54